# 1. Imports

In [2]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 500)
pd.options.plotting.backend = "plotly"

# 2. Data

### 2.1 Données de validation de titres.

In [ ]:

# 4 fichiers csv de validations de titres (1 par trimestre) pour 2025 à combiner

# Chemin du répertoire où se trouvent les fichiers csv à combiner
folder_path_validations = os.path.join("..","data","validations_multiples")

# Récuperer tous les fichiers csv dans ce répertoire
validations = glob.glob(os.path.join(folder_path_validations, "*.csv"))

# List comprehension pour lire et concaténer
validations_df = pd.concat((pd.read_csv(f, sep=";") for f in validations), ignore_index=True)

print(f"Successfully stacked {len(validations)} CSV files into 'validations.csv'!")



Successfully stacked 4 CSV files into 'validations.csv'!


### 2.2 Liste et emplacement des gares / stations.

#### 2.2.1. Liste des stations généralisée.

In [ ]:
# Liste des stations généralisée (une station peut regrouper plusieurs lignes de transport associées)
filepath_stations_g = os.path.join("..","data","stations","emplacement-des-gares-idf-data-generalisee.csv")
stations_g_df = pd.read_csv(filepath_stations_g, sep=";")

#### 2.2.2. Liste des stations.

In [ ]:
# Liste des stations (une station par ligne de transport associée)
filepath_stations = os.path.join("..","data","stations","emplacement-des-gares-idf.csv")
stations_df = pd.read_csv(filepath_stations,sep=";")

# 3. EDA

### 3.1. Données de validation de titres.

#### 3.1.1. Vue globale des données.

In [46]:
validations_df.head()

,jour,code_stif_trns,code_stif_res,code_stif_arret,id_zdc,libelle_arret,categorie_titre,nb_vald
0,2025-10-31,800.0,803,454,69662.0,SAULES,Autres titres,1.0
1,2025-10-31,800.0,803,454,69662.0,SAULES,Forfaits courts,188.0
2,2025-10-31,800.0,803,788,60987.0,SAVIGNY SUR ORG,Imagine R,1263.0
3,2025-10-31,800.0,803,793,59842.0,SERMAISE,Amethyste,1.0
4,2025-10-31,800.0,803,793,59842.0,SERMAISE,Forfait Navigo,42.0


Colonnes interessantes pour l'analyse : jour, id_zdc pour faire les jointures, categorie_titre, nb_vald.

In [7]:
print(validations_df.shape)
print(f"Données de validation : Nombre de ligne : {validations_df.shape[0]}, Nombre de colonnes : {validations_df.shape[1]} ")

(1892453, 8)
Données de validation : Nombre de ligne : 1892453, Nombre de colonnes : 8 


In [8]:
validations_df.columns

Index(['jour', 'code_stif_trns', 'code_stif_res', 'code_stif_arret', 'id_zdc',
       'libelle_arret', 'categorie_titre', 'nb_vald'],
      dtype='object')

In [9]:
validations_df.describe()

,code_stif_trns,id_zdc,nb_vald
count,1.892453e+06,1.890831e+06,1.892453e+06
mean,5.020467e+02,9.026463e+04,1.140139e+03
std,3.472100e+02,8.951037e+04,3.084966e+03
min,1.000000e+02,-1.000000e+00,1.000000e+00
25%,1.000000e+02,6.653500e+04,4.400000e+01
50%,8.000000e+02,7.115000e+04,2.150000e+02
75%,8.000000e+02,7.192000e+04,1.016000e+03
max,8.100000e+02,9.999990e+05,9.997000e+04


In [10]:
validations_df.describe(include=object)

,jour,code_stif_res,code_stif_arret,libelle_arret,categorie_titre
count,1892453,1892453,1892453,1892453,1892453
unique,365,18,787,779,8
top,2025-11-04,110,48093,GARE DE LYON,Forfait Navigo
freq,5365,806414,8905,7665,284117


In [24]:
validations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1892453 entries, 0 to 1892452
Data columns (total 8 columns):
 #   Column           Dtype  
---  ------           -----  
 0   jour             object 
 1   code_stif_trns   float64
 2   code_stif_res    object 
 3   code_stif_arret  object 
 4   id_zdc           float64
 5   libelle_arret    object 
 6   categorie_titre  object 
 7   nb_vald          float64
dtypes: float64(3), object(5)
memory usage: 115.5+ MB


In [47]:
validations_df["categorie_titre"].value_counts()

categorie_titre
Forfait Navigo                  284117
Imagine R                       282766
Forfaits courts                 280786
Amethyste                       270264
NON DEFINI                      248620
Autres titres                   244447
Contrat Solidarité Transport    210239
Contrat Solidarite Transport     71214
Name: count, dtype: int64

Doublon catégorie pour "Solidarité" et "Solidarite"  
NON DEFINI désigne une anomalie chez idfm. Ne pas grouper avec "Autres titres".

#### 3.1.2. Data cleaning.

In [50]:
# Correction "categorie_titre": renommer "Contrat Solidarité Transport" par "Contrat Solidarite Transport"
validations_df["categorie_titre"] = validations_df["categorie_titre"].replace("Contrat Solidarité Transport", "Contrat Solidarite Transport")

In [51]:
validations_df["categorie_titre"].value_counts()

categorie_titre
Forfait Navigo                  284117
Imagine R                       282766
Contrat Solidarite Transport    281453
Forfaits courts                 280786
Amethyste                       270264
NON DEFINI                      248620
Autres titres                   244447
Name: count, dtype: int64

In [ ]:
# Valeurs manquantes.
validations_df.isna().sum()

jour                  0
code_stif_trns        0
code_stif_res         0
code_stif_arret       0
id_zdc             1622
libelle_arret         0
categorie_titre       0
nb_vald               0
dtype: int64

1622 valeurs manquantes sur 2 millions pour la colonne id_zdc

In [ ]:
# A quoi correspondent les valeurs manquantes ?
validations_df[validations_df["id_zdc"].isnull()]

,jour,code_stif_trns,code_stif_res,code_stif_arret,id_zdc,libelle_arret,categorie_titre,nb_vald
481190,2025-09-22,100.0,ND,ND,NaN,Inconnu,Amethyste,155.0
484136,2025-09-19,100.0,ND,ND,NaN,Inconnu,Amethyste,177.0
484137,2025-09-19,100.0,ND,ND,NaN,Inconnu,Contrat Solidarité Transport,683.0
484138,2025-09-19,100.0,ND,ND,NaN,Inconnu,Imagine R,1388.0
485797,2025-09-24,100.0,ND,ND,NaN,Inconnu,Forfaits courts,537.0
...,...,...,...,...,...,...,...,...
1888840,2025-06-27,100.0,ND,ND,NaN,Inconnu,Forfaits courts,5.0
1888841,2025-06-27,760.0,760,00490909,NaN,Aéroport d'Orly,Contrat Solidarité Transport,841.0
1889842,2025-06-30,760.0,760,00490909,NaN,Aéroport d'Orly,Imagine R,1493.0
1889843,2025-06-30,760.0,760,00490909,NaN,Aéroport d'Orly,NON DEFINI,366.0


In [28]:
# Doublons
validations_df.duplicated().sum()

np.int64(0)

Pas de doublons

### 3.2. Liste et emplacement des gares / stations.

#### 3.2.1. Liste des stations généralisée.

##### 3.2.1.1. Vue globale des donnés.

In [32]:
stations_g_df.head()

,geo_point_2d,geo_shape,objectid_1,codeunique,nom_long,nom_so_gar,nom_su_gar,id_ref_zdc,nom_zdc,id_ref_zda,nom_zda,idrefliga,idrefligc,res_com,mode,train,rer,metro,tramway,val,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,x,y
0,"48.88218913760796, 2.70601418111842","{""coordinates"": [2.70601418111842, 48.88218913...",5,3,Lagny-Thorigny,NaN,NaN,68494,Lagny - Thorigny,427872,Lagny - Thorigny,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,678439.8038,6.864726e+06
1,"48.83964551726303, 2.655130086641314","{""coordinates"": [2.655130086641314, 48.8396455...",7,7,Torcy,NaN,NaN,68129,Torcy,43207,Torcy,A01856,C01742,RER A,RER,0,1,0,0,0,0,0,0,0,0,RATP,1,0,674687.4504,6.860010e+06
2,"48.79565482051668, 2.6503876698084805","{""coordinates"": [2.650387669808481, 48.7956548...",9,10,Roissy-en-Brie,NaN,NaN,67839,Roissy-en-Brie,46568,Roissy-en-Brie,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,674317.7148,6.855121e+06
3,"48.77077891616031, 2.690252782201096","{""coordinates"": [2.690252782201096, 48.7707789...",10,11,Ozoir-la-Ferrière,NaN,NaN,462934,Ozoir-la-Ferrière,462901,Ozoir-la-Ferrière,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,677235.3132,6.852342e+06
4,"48.960226490735955, 2.9500603827528087","{""coordinates"": [2.950060382752809, 48.9602264...",13,16,Trilport,NaN,NaN,68984,Trilport,47962,Trilport,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,696343.0315,6.873365e+06


Colonnes interessantes pour l'analyse :  
geo_point_2d pour carte  
id_ref_zdc pour les jointures  
nom_zdc  
res_com  
mode  
train, rer, metro, tramway, val pour identifier les modes  
tertrain, terrer, termetro, tertram, terval pour voir l'impact des terminus  
exploitant  
principal 

In [45]:
print(stations_g_df.shape)
print(f"Données de validation : Nombre de ligne : {stations_g_df.shape[0]}, Nombre de colonnes : {stations_g_df.shape[1]} ")

(999, 30)
Données de validation : Nombre de ligne : 999, Nombre de colonnes : 30 


In [34]:
stations_g_df.columns

Index(['geo_point_2d', 'geo_shape', 'objectid_1', 'codeunique', 'nom_long',
       'nom_so_gar', 'nom_su_gar', 'id_ref_zdc', 'nom_zdc', 'id_ref_zda',
       'nom_zda', 'idrefliga', 'idrefligc', 'res_com', 'mode', 'train', 'rer',
       'metro', 'tramway', 'val', 'tertrain', 'terrer', 'termetro', 'tertram',
       'terval', 'exploitant', 'idf', 'principal', 'x', 'y'],
      dtype='object')

In [26]:
stations_g_df.describe().T

,count,mean,std,min,25%,50%,75%,max
objectid_1,999.0,5.361041e+02,336.213435,-1.000000,2.265000e+02,5.470000e+02,8.345000e+02,1.101000e+03
codeunique,999.0,3.987828e+04,52563.103408,1.000000,2.585000e+02,5.210000e+02,1.070280e+05,1.171150e+05
id_ref_zdc,999.0,1.108205e+05,117408.947771,59403.000000,6.733850e+04,7.122300e+04,7.246600e+04,4.909190e+05
id_ref_zda,999.0,1.214172e+05,160126.428183,42142.000000,4.328400e+04,4.563000e+04,5.401650e+04,4.940200e+05
train,999.0,2.692693e-01,0.443802,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
rer,999.0,2.402402e-01,0.427443,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
metro,999.0,3.213213e-01,0.467218,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
tramway,999.0,2.742743e-01,0.446371,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
val,999.0,1.001001e-02,0.099598,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
tertrain,999.0,3.503504e-02,0.183960,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00


In [27]:
stations_g_df.describe(include=object).T

,count,unique,top,freq
geo_point_2d,999,999,"48.88218913760796, 2.70601418111842",1
geo_shape,999,999,"{""coordinates"": [2.70601418111842, 48.88218913...",1
nom_long,999,997,Saint-Fargeau,2
nom_so_gar,68,65,Hôtel de Ville,2
nom_su_gar,29,12,VITRY-SUR-SEINE,6
nom_zdc,999,990,NC,3
nom_zda,999,994,Saint-Fargeau,2
idrefliga,999,188,A01842,49
idrefligc,999,190,C01728,49
res_com,999,186,RER D,49


In [12]:
stations_g_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 30 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   geo_point_2d  999 non-null    object 
 1   geo_shape     999 non-null    object 
 2   objectid_1    999 non-null    int64  
 3   codeunique    999 non-null    int64  
 4   nom_long      999 non-null    object 
 5   nom_so_gar    68 non-null     object 
 6   nom_su_gar    29 non-null     object 
 7   id_ref_zdc    999 non-null    int64  
 8   nom_zdc       999 non-null    object 
 9   id_ref_zda    999 non-null    int64  
 10  nom_zda       999 non-null    object 
 11  idrefliga     999 non-null    object 
 12  idrefligc     999 non-null    object 
 13  res_com       999 non-null    object 
 14  mode          999 non-null    object 
 15  train         999 non-null    int64  
 16  rer           999 non-null    int64  
 17  metro         999 non-null    int64  
 18  tramway       999 non-null    

##### 3.2.1.2. Data cleaning.

In [ ]:
# Valeurs manquantes.
stations_g_df.isna().sum()

geo_point_2d      0
geo_shape         0
objectid_1        0
codeunique        0
nom_long          0
nom_so_gar      931
nom_su_gar      970
id_ref_zdc        0
nom_zdc           0
id_ref_zda        0
nom_zda           0
idrefliga         0
idrefligc         0
res_com           0
mode              0
train             0
rer               0
metro             0
tramway           0
val               0
tertrain          0
terrer            0
termetro          0
tertram           0
terval            0
exploitant        0
idf               0
principal         0
x                 0
y                 0
dtype: int64

Pas de valeurs manquantes sauf pour nom_so_gar (nom sous gare) et nom_su_gar (nom sur gare) qui est normal 

In [29]:
# Doublons
stations_g_df.duplicated().sum()

np.int64(0)

Pas de doublons.

#### 3.2.2. Liste des stations.

##### 3.2.2.1. Vue globale des données.

In [29]:
stations_df.head()

,geo_point_2d,geo_shape,objectid_1,id_gares,nom_gares,nom_so_gar,nom_su_gar,id_ref_zdc,nom_zdc,id_ref_zda,nom_zda,idrefliga,idrefligc,res_com,indice_lig,mode,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,x,y,picto,nom_iv
0,"48.667901961049886, 2.5475480458143664","{""coordinates"": [2.547548045814366, 48.6679019...",91,196,Combs-la-Ville - Quincy,NaN,NaN,62558,Combs-la-Ville - Quincy,45771,Combs-la-Ville - Quincy,A01842,C01728,RER D,D,RER,0,0,0,0,0,SNCF,1,0,666681.8714,6.840955e+06,https://data.iledefrance-mobilites.fr/api/expl...,Combs-la-Ville - Quincy
1,"48.613427913594734, 2.4729336504587436","{""coordinates"": [2.472933650458744, 48.6134279...",94,207,Corbeil-Essonnes,NaN,NaN,60309,Corbeil-Essonnes,43115,Corbeil-Essonnes,A01842,C01728,RER D,D,RER,0,0,0,0,0,SNCF,1,1,661146.9910,6.834933e+06,https://data.iledefrance-mobilites.fr/api/expl...,Corbeil-Essonnes
2,"48.843772576851464, 2.3739164698064985","{""coordinates"": [2.373916469806499, 48.8437725...",128,307,Gare de Lyon,NaN,NaN,73626,Gare de Lyon,470195,Gare de Lyon,A01842,C01728,RER D,D,RER,0,0,0,0,0,SNCF,1,1,654051.0804,6.860596e+06,https://data.iledefrance-mobilites.fr/api/expl...,Gare de Lyon
3,"49.03318276903128, 2.478033365115082","{""coordinates"": [2.478033365115082, 49.0331827...",190,452,Les Noues,NaN,NaN,66650,Les Noues,47968,Les Noues,A01842,C01728,RER D,D,RER,0,0,0,0,0,SNCF,1,0,661831.4373,6.881603e+06,https://data.iledefrance-mobilites.fr/api/expl...,Les Noues
4,"49.13828637527201, 2.4901789854211955","{""coordinates"": [2.490178985421196, 49.1382863...",242,957,Orry-la-Ville-Coye-la-Forêt,NaN,NaN,411422,Orry-la-Ville - Coye,411421,Orry-la-Ville - Coye,A01842,C01728,RER D,D,RER,0,0,0,0,0,SNCF,0,1,662795.0403,6.893287e+06,https://data.iledefrance-mobilites.fr/api/expl...,Orry-la-Ville-Coye-la-Forêt


Colonnes interessantes pour l'analyse :  
geo_point_2d pour carte  
nom_gare  
id_ref_zdc pour les jointures  
res_com  
mode  
train, rer, metro, tramway, val pour identifier les modes  
tertrain, terrer, termetro, tertram, terval pour voir l'impact des terminus  
exploitant  
idf si gare en idf ou non  
principal 

In [46]:
print(stations_df.shape)
print(f"Données de validation : Nombre de ligne : {stations_df.shape[0]}, Nombre de colonnes : {stations_df.shape[1]} ")

(1240, 28)
Données de validation : Nombre de ligne : 1240, Nombre de colonnes : 28 


In [31]:
stations_df.columns

Index(['geo_point_2d', 'geo_shape', 'objectid_1', 'id_gares', 'nom_gares',
       'nom_so_gar', 'nom_su_gar', 'id_ref_zdc', 'nom_zdc', 'id_ref_zda',
       'nom_zda', 'idrefliga', 'idrefligc', 'res_com', 'indice_lig', 'mode',
       'tertrain', 'terrer', 'termetro', 'tertram', 'terval', 'exploitant',
       'idf', 'principal', 'x', 'y', 'picto', 'nom_iv'],
      dtype='object')

In [16]:
stations_df.describe()

,objectid_1,id_gares,id_ref_zdc,id_ref_zda,idf,principal,x,y
count,1240.000000,1240.000000,1240.000000,1240.000000,1240.000000,1240.000000,1240.000000,1.240000e+03
mean,673.658871,675.323387,108216.090323,143790.666935,0.958065,0.112097,651170.248968,6.833170e+06
std,443.392990,426.578218,114065.562359,177141.590123,0.200523,0.315613,17547.499217,4.277332e+05
min,-1.000000,0.000000,59403.000000,42142.000000,0.000000,0.000000,564673.761000,1.149219e+05
25%,277.750000,321.750000,68306.500000,43404.000000,1.000000,0.000000,646031.962275,6.856525e+06
50%,668.500000,647.500000,71298.000000,45762.000000,1.000000,0.000000,651624.635500,6.862556e+06
75%,1020.250000,998.250000,72389.250000,58970.750000,1.000000,0.000000,655641.924125,6.867296e+06
max,1497.000000,1874.000000,490919.000000,494022.000000,1.000000,1.000000,738278.993500,6.925576e+06


In [17]:
stations_df.describe(include="object")

,geo_point_2d,geo_shape,nom_gares,nom_so_gar,nom_su_gar,nom_zdc,nom_zda,idrefliga,idrefligc,res_com,indice_lig,mode,tertrain,terrer,termetro,tertram,terval,exploitant,picto,nom_iv
count,1240,1240,1240,91,29,1240,1240,1240,1240,1240,1240,1240,1240,1240,1240,1240,1240,1240,1132,1240
unique,1180,1180,1014,70,10,993,1014,50,51,50,36,7,12,6,17,15,5,9,39,1014
top,"48.795138179363484, 2.135156224100022","{""coordinates"": [2.135156224100022, 48.7951381...",Gare du Nord,Grande Arche,VITRY-SUR-SEINE,Gare Saint-Lazare,République,A01840,C01727,RER C,C,METRO,0,0,0,0,0,RATP,https://data.iledefrance-mobilites.fr/api/expl...,Gare du Nord
freq,4,4,6,6,6,7,5,82,75,75,75,405,1200,1218,1205,1209,1233,665,82,6


In [13]:
stations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1240 entries, 0 to 1239
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   geo_point_2d  1240 non-null   object 
 1   geo_shape     1240 non-null   object 
 2   objectid_1    1240 non-null   int64  
 3   id_gares      1240 non-null   int64  
 4   nom_gares     1240 non-null   object 
 5   nom_so_gar    91 non-null     object 
 6   nom_su_gar    29 non-null     object 
 7   id_ref_zdc    1240 non-null   int64  
 8   nom_zdc       1240 non-null   object 
 9   id_ref_zda    1240 non-null   int64  
 10  nom_zda       1240 non-null   object 
 11  idrefliga     1240 non-null   object 
 12  idrefligc     1240 non-null   object 
 13  res_com       1240 non-null   object 
 14  indice_lig    1240 non-null   object 
 15  mode          1240 non-null   object 
 16  tertrain      1240 non-null   object 
 17  terrer        1240 non-null   object 
 18  termetro      1240 non-null 

##### 3.2.2.2. Data cleaning.

In [ ]:
# Valeurs manquantes.
stations_df.isna().sum()

geo_point_2d       0
geo_shape          0
objectid_1         0
id_gares           0
nom_gares          0
nom_so_gar      1149
nom_su_gar      1211
id_ref_zdc         0
nom_zdc            0
id_ref_zda         0
nom_zda            0
idrefliga          0
idrefligc          0
res_com            0
indice_lig         0
mode               0
tertrain           0
terrer             0
termetro           0
tertram            0
terval             0
exploitant         0
idf                0
principal          0
x                  0
y                  0
picto            108
nom_iv             0
dtype: int64

Pas de valeurs manquantes sauf pour nom_so_gar (nom sous gare) et nom_su_gar (nom sur gare) qui est normal et picto qu'on n'utilisera pas..

In [30]:
# Doublons.
stations_df.duplicated().sum()

np.int64(0)

Pas de doublons.

# 4. Export

Exporter les fichiers en format csv pour SQL Big Query.

In [ ]:
# Sauvegarder les données dans un nouveau csv SQL Big Query.

clean_validations_filepath = os.path.join('..','data','clean_validations','validations.csv')
validations_df.to_csv(clean_validations_filepath, index=False)

clean_stations_g_filepath = os.path.join('..','data','clean_stations','stations_g.csv')
stations_g_df.to_csv(clean_stations_g_filepath, index=False)

clean_stations_filepath = os.path.join('..','data','clean_stations','stations.csv')
stations_df.to_csv(clean_stations_filepath, index=False)